# Workshop: Machine Learning for Aquatic Remote Sensing
**Event:** International Science Council (ISC) SCOR Workshop on Satellite Remote Sensing <br>
**Location:** Department of Marine Sciences, Berhampur University, India <br>
**Date:** Dec 2025 <br>
**Instructor:** Chintan B. Maniyar, PhD Candidate, University of Georgia (chintanmaniyar@uga.edu) <br>

---
**Note on Usage:**
This notebook was developed specifically for educational purposes within the SCOR workshop curriculum. The code and workflows demonstrate the application of Machine Learning to aquatic remote sensing data. While compliant with scientific best practices, users should rigorously validate these models before applying them to operational or published research.

### Import Packages

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append(r'./../')
import utils # this is my custom toolkit for visualization

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn import tree
from xgboost import XGBRegressor
from sklearn.svm import SVR

import joblib

### Start with $R_{rs}$ data

We will read the data, just like we did in the previous notebook

In [ ]:
df_rrs = pd.read_csv('./../data/rrs_chla_olci.csv', index_col=0)
df_rrs = df_rrs.drop(['761', '764', '767', '778'], axis=1)
df_rrs

Now, we will segregate our *features* or predictor variables, and the target. Here, the features will be all available bands of Sentinel-3 OLCI, and the target will be Chl-*a* concentration. In ML terminology, the set of features is denoted by **X** and the target variable is denoted by **y**.

In [ ]:
X = df_rrs.drop('Chla', axis=1).to_numpy() # these are all the bands
y = df_rrs['Chla'].to_numpy().reshape(-1,1) # this is chl-a concentration
X.shape, y.shape # number of samples in X and y (sanity check)

#### Train-test data split

This is one of the most important concepts in ML, or any kind of modeling for that matter. If you have built statistical models, you probably call this step as "cal-val" split. Train data, or cal data, is the chunk of data on which you *fit* your model. And the test data, or val data, is the chunk of your total dataset on which you *assess* the model fit.

In total, we have $N=1502$ data points. Let's do a random train-test split in a way that 70% of our data is used for training (cal data), and 30% of the data is used for testing (val data).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=777) # different students  try different value for random_state
X_train.shape, y_train.shape, X_test.shape, y_test.shape # notice the number of samples in each partition now. What do you observe?

Notice the size of ```X_train``` and ```X_test``` now. It is 1201 and 301 respectively, which is 70% and 30% of the total data ($N=1502$) respectively. Same goes for ```y_train``` and ```y_test``` respectively. 

> *Question*: If you look at the shape of ```X```, ```(1502, 15)```.  Same for ```X_train``` and ```X_test```. However, in case of ```y```, ```y_train``` and ```y_test```, you see ```(1502, 1)```. Why is that? What does $15$ and $1$ signify?

Now, let is look at the split from a quantitatve view. How does the target in training data and testing data vary?

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,5))
sns.histplot(y_train, kde=True, ax=ax[0], legend=False)
sns.histplot(y_train, kde=True, ax=ax[1], log_scale=True,legend=False)
ax[0].set_xlabel('Chl-a [$mg/m^3$]')
ax[1].set_xlabel('Chl-a [$mg/m^3$]')
ax[0].set_title(rf'$N={len(y_train)}$' + '\nLinear Scale')
ax[1].set_title(rf'$N={len(y_train)}$' + '\nLog Scale')
fig.suptitle('Train Data (y_train)')

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,5))
sns.histplot(y_test, kde=True, ax=ax[0], legend=False)
sns.histplot(y_test, kde=True, ax=ax[1], log_scale=True,legend=False)
ax[0].set_xlabel('Chl-a [$mg/m^3$]')
ax[1].set_xlabel('Chl-a [$mg/m^3$]')
ax[0].set_title(rf'$N={len(y_test)}$' + '\nLinear Scale')
ax[1].set_title(rf'$N={len(y_test)}$' + '\nLog Scale')
fig.suptitle('Test Data (y_test)')

> *Question*: What is the ```random_state``` parameter in the  ```train_test_split()``` function? What happens if you change it? Why do we need it?

## Build a Linear Model with this data

In [ ]:
# fir the model using training data
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
# save model to use with satellite image
joblib.dump(model, "./../models/linear_model_rrs.joblib")

In [ ]:
model.score(X_train, y_train)

In [ ]:
# evaluate the model using testing data
y_pred = model.predict(X_test)

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(18, 6))

utils.get_validation_plot(x=y_test.ravel(), y=y_pred.ravel(), metrics=['r2', 'nrmse', 'bias', 'mape'], ax_val=ax[0], log_norm=True,
                          xlabel_val="Measured Chla [$mg/m^3$]", ylabel_val="Modeled Chla [$mg/m^3$]", xlabel_res="$\Delta\ Chla\ [mg/m^3]$",
                          title_val= "Model validation plot, Log-scale", min_threshold=0.001, color_val='black')

utils.get_validation_plot(x=y_test.ravel(), y=y_pred.ravel(), metrics=['r2', 'nrmse', 'bias', 'mape'], ax_val=ax[1],
                          xlabel_val="Measured Chla [$mg/m^3$]", ylabel_val="Modeled Chla [$mg/m^3$]", xlabel_res="$\Delta\ Chla\ [mg/m^3]$",
                          title_val= "Model validation plot, Linear-scale", title_res="Histogram of Residuals", ax_res=ax[2], color_val='black',
                          min_threshold=0.001, residuals=True)

## Build Machine Learning Models with this data

Now, we will use the same dataset and apply various ML models to see if they do better or worse than the standard regression model we just saw!

### 1. Random Forest (Tree based)

In [ ]:
# fit the model using our train data
model = RandomForestRegressor(random_state=72) # What does the random state do? Change it and find out!
model.fit(X_train, y_train.ravel())

In [ ]:
# save model to use with satellite image
joblib.dump(model, "./../models/random_forest_rrs.joblib")

In [ ]:
# evaluate the model o the test data
y_pred = model.predict(X_test)

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(18, 6))

utils.get_validation_plot(x=y_test.ravel(), y=y_pred.ravel(), metrics=['r2', 'nrmse', 'bias', 'mape'], ax_val=ax[0], log_norm=True,
                          xlabel_val="Measured Chla [$mg/m^3$]", ylabel_val="Modeled Chla [$mg/m^3$]", xlabel_res="$\Delta\ Chla\ [mg/m^3]$",
                          title_val= "Model validation plot, Log-scale", min_threshold=0.001, color_val='black')

utils.get_validation_plot(x=y_test.ravel(), y=y_pred.ravel(), metrics=['r2', 'nrmse', 'bias', 'mape'], ax_val=ax[1],
                          xlabel_val="Measured Chla [$mg/m^3$]", ylabel_val="Modeled Chla [$mg/m^3$]", xlabel_res="$\Delta\ Chla\ [mg/m^3]$",
                          title_val= "Model validation plot, Linear-scale", title_res="Histogram of Residuals", ax_res=ax[2], color_val='black',
                          min_threshold=0.001, residuals=True)

> *Question*: How did this model do, compared to the linear model? Is it better or worse? Why do you think that is?

Now, let us look at something called "feature importance", which is a part of making machine learning models more "explainable". This is one of the most important and hot sub disciplines of ML/AI right now, especially in applied sciences, and is called **XAI** or **Explainable AI**. 

In [ ]:
feature_importance_df = pd.DataFrame({
    'Feature': df_rrs.drop("Chla", axis=1).columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

In [ ]:
feature_importance_df = feature_importance_df.set_index("Feature")

In [ ]:
feature_importance_df.plot(kind='bar',)

> This plot suggests that the band at 620nm in OLCI is the most important feature. Does that make sense biogeochemically? Isn't 620nm a distinct characteristic for Phycocyanin, and not Chlorophyll? What does this tell you about ML, and its scientific grounding?

Now, let us try to visualize one of the trees in this random forest model, to see how the "learning" actually happens.

In [ ]:
estimator = model.estimators_[0] # Get the first tree

plt.figure(figsize=(40, 20))
tree.plot_tree(estimator, 
               feature_names=feature_importance_df.index,
               max_depth=4,
               filled=True, 
               rounded=True,
               precision=2)
plt.title("Single Decision Tree from Random Forest Regressor")
plt.show()

### 2. Extreme Gradient Boosting (XGB) Regression (Tree-based)

In [ ]:
# fit the model using our train data
model = XGBRegressor(random_state=42)
model.fit(X_train, y_train)

In [ ]:
# save model to use with satellite image
joblib.dump(model, "./../models/xgb_rrs.joblib")

In [ ]:
# evaluate the model o the test data
y_pred = model.predict(X_test)

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(18, 6))

utils.get_validation_plot(x=y_test.ravel(), y=y_pred.ravel(), metrics=['r2', 'nrmse', 'bias', 'mape'], ax_val=ax[0], log_norm=True,
                          xlabel_val="Measured Chla [$mg/m^3$]", ylabel_val="Modeled Chla [$mg/m^3$]", xlabel_res="$\Delta\ Chla\ [mg/m^3]$",
                          title_val= "Model validation plot, Log-scale", min_threshold=0.001, color_val='black')

utils.get_validation_plot(x=y_test.ravel(), y=y_pred.ravel(), metrics=['r2', 'nrmse', 'bias', 'mape'], ax_val=ax[1],
                          xlabel_val="Measured Chla [$mg/m^3$]", ylabel_val="Modeled Chla [$mg/m^3$]", xlabel_res="$\Delta\ Chla\ [mg/m^3]$",
                          title_val= "Model validation plot, Linear-scale", title_res="Histogram of Residuals", ax_res=ax[2], color_val='black',
                          min_threshold=0.001, residuals=True)

> *Question*: How did this model do, compared to Random Forest? Look at the different metrics and talk about it.

Now, let us look at the feature important of XGB regression. Which feature helped it the most?

In [ ]:
feature_importance_df = pd.DataFrame({
    'Feature': df_rrs.drop("Chla", axis=1).columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

In [ ]:
feature_importance_df = feature_importance_df.set_index("Feature")

In [ ]:
feature_importance_df.plot(kind='bar',)

>*Question*: Is the feature importance for XGB same as that of Random Forest? Or is it different? Why do you think that is? For the current model, is the most important feature 761nm? 

### 3. Support Vector Machine

In [ ]:
model = SVR(kernel='rbf', C=300, gamma=.6) # these are the hyperparameters we talked about
model.fit(X_train, y_train.ravel())

In [ ]:
# save model to use with satellite image
joblib.dump(model, "./../models/svm_rrs.joblib")

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(18, 6))

utils.get_validation_plot(x=y_test.ravel(), y=y_pred.ravel(), metrics=['r2', 'nrmse', 'bias', 'mape'], ax_val=ax[0], log_norm=True,
                          xlabel_val="Measured Chla [$mg/m^3$]", ylabel_val="Modeled Chla [$mg/m^3$]", xlabel_res="$\Delta\ Chla\ [mg/m^3]$",
                          title_val= "Model validation plot, Log-scale", min_threshold=0.001, color_val='black')

utils.get_validation_plot(x=y_test.ravel(), y=y_pred.ravel(), metrics=['r2', 'nrmse', 'bias', 'mape'], ax_val=ax[1],
                          xlabel_val="Measured Chla [$mg/m^3$]", ylabel_val="Modeled Chla [$mg/m^3$]", xlabel_res="$\Delta\ Chla\ [mg/m^3]$",
                          title_val= "Model validation plot, Linear-scale", title_res="Histogram of Residuals", ax_res=ax[2], color_val='black',
                          min_threshold=0.001, residuals=True)

> *Question*: How did SVM regression do out of all the ML models? Why do you think that is? Recall what we talked about in last session about SVM and their drawbacks.

## Build ML models with BR-LH (feature engineered data)

In [ ]:
df_br_lh = pd.read_csv('./../data/br_lh_chla_olci.csv', index_col=0)
df_br_lh

In [ ]:
X = df_br_lh.drop('Chla', axis=1).to_numpy() # these are all the bands
y = df_br_lh['Chla'].to_numpy().reshape(-1,1) # this is chl-a concentration
X.shape, y.shape # number of samples in X and y (sanity check)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=777) # different students  try different value for random_state
X_train.shape, y_train.shape, X_test.shape, y_test.shape # notice the number of samples in each partition now. What do you observe?

### 1. Random Forest Regression

In [ ]:
# fit the model using our train data
model = RandomForestRegressor(random_state=72) # What does the random state do? Change it and find out!
model.fit(X_train, y_train.ravel())

In [ ]:
# evaluate the model o the test data
y_pred = model.predict(X_test)

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(18, 6))

utils.get_validation_plot(x=y_test.ravel(), y=y_pred.ravel(), metrics=['r2', 'nrmse', 'bias', 'mape'], ax_val=ax[0], log_norm=True,
                          xlabel_val="Measured Chla [$mg/m^3$]", ylabel_val="Modeled Chla [$mg/m^3$]", xlabel_res="$\Delta\ Chla\ [mg/m^3]$",
                          title_val= "Model validation plot, Log-scale", min_threshold=0.001, color_val='black')

utils.get_validation_plot(x=y_test.ravel(), y=y_pred.ravel(), metrics=['r2', 'nrmse', 'bias', 'mape'], ax_val=ax[1],
                          xlabel_val="Measured Chla [$mg/m^3$]", ylabel_val="Modeled Chla [$mg/m^3$]", xlabel_res="$\Delta\ Chla\ [mg/m^3]$",
                          title_val= "Model validation plot, Linear-scale", title_res="Histogram of Residuals", ax_res=ax[2], color_val='black',
                          min_threshold=0.001, residuals=True)

In [ ]:
feature_importance_df = pd.DataFrame({
    'Feature': df_br_lh.drop("Chla", axis=1).columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

In [ ]:
feature_importance_df = feature_importance_df.set_index("Feature")

In [ ]:
feature_importance_df.plot(kind='bar',)

> ## *Question*: Can you extend the code to use BR/LH instead of $R_{rs}$ for the other 2 ML models?

### 2. XGB Regression

In [ ]:
# your code here

### 3. SVM Regression

In [ ]:
# your code here